# Image Edit — ComfyUI

İki workflow:
- **Qwen Image Edit**: Prompt ile görsel düzenleme (obje değiştir, stil transfer)
- **FLUX.2-dev**: Yüksek kaliteli text-to-image + turbo LoRA

## Colab Secrets
- `CF_TUNNEL_TOKEN`, `HF_TOKEN`

## Kullanım
A: Kurulum → B: Model indir → C: Başlat

---
# A) Kurulum

In [ ]:
import os
import subprocess

import torch

if not torch.cuda.is_available():
    raise RuntimeError('GPU bulunamadı!')
gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'\u2705 GPU: {gpu_name} ({gpu_mem:.1f} GB)')

os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('\u2705 HF_TOKEN')
except Exception:
    print('\u26a0\ufe0f HF_TOKEN yok')

# ComfyUI
COMFY_DIR = '/content/ComfyUI'
CUSTOM_NODES = f'{COMFY_DIR}/custom_nodes'

if not os.path.exists(COMFY_DIR):
    print('\U0001f4e6 ComfyUI...')
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}
    !pip install -q -r {COMFY_DIR}/requirements.txt
else:
    print('\u2705 ComfyUI mevcut')

# Custom Node'lar
NODES = {
    'ComfyUI-VideoHelperSuite': 'https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git',
    'ComfyUI-Manager': 'https://github.com/ltdrdata/ComfyUI-Manager.git',
    'ComfyUI-Impact-Pack': 'https://github.com/ltdrdata/ComfyUI-Impact-Pack.git',
    'rgthree-comfy': 'https://github.com/rgthree/rgthree-comfy.git',
}

for name, url in NODES.items():
    node_dir = f'{CUSTOM_NODES}/{name}'
    if not os.path.exists(node_dir):
        print(f'  \u2193 {name}')
        !git clone --depth 1 {url} {node_dir}
        req_file = f'{node_dir}/requirements.txt'
        if os.path.exists(req_file):
            !pip install -q -r {req_file}
    else:
        print(f'  \u2713 {name}')

# Cloudflare Tunnel
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

print('\n\u2705 Kurulum tamam')

---
# B) Model İndir

In [ ]:
MODELS_DIR = f'{COMFY_DIR}/models'

# ╔══════════════════════════════════════════════════════════════╗
# ║  WORKFLOW SEÇ — değiştirip B'yi tekrar çalıştır            ║
# ║  Eski modeller otomatik silinir, ortak kalanlar kalır      ║
# ╚══════════════════════════════════════════════════════════════╝
WORKFLOW = 'qwen_edit_2511'  # 'qwen_edit_2511' | 'qwen_controlnet' | 'qwen_inpainting' | 'flux2_dev' | 'flux2_klein_kv' | 'flux2_klein_edit'


def hf_download(repo, filename, dest_dir):
    basename = filename.split('/')[-1]
    dest = f'{dest_dir}/{basename}'
    if os.path.exists(dest):
        print(f'  \u2713 {basename} (mevcut)')
        return
    print(f'  \u2193 {basename}...')
    os.makedirs(dest_dir, exist_ok=True)
    from huggingface_hub import hf_hub_download
    try:
        path = hf_hub_download(repo_id=repo, filename=filename, local_dir='/content/hf_cache')
        import shutil
        shutil.move(path, dest)
        print(f'  \u2705 {basename}')
    except Exception as e:
        print(f'  \u274c Başarısız: {e}')


def remove_if_exists(path):
    if os.path.exists(path):
        os.remove(path)
        print(f'  \U0001f5d1 Silindi: {os.path.basename(path)}')


# ─── Workflow'a özel modeller + temizlik ───

# Tüm workflow'lara özel dosyalar (sil listesi)
QWEN_EDIT_FILES = ['qwen_image_edit_2511_bf16.safetensors', 'Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors']
QWEN_CN_FILES = ['Qwen-Image-2512-Fun-Controlnet-Union-2602.safetensors', 'qwen_image_2512_fp8_e4m3fn.safetensors', 'Qwen-Image-Lightning-4steps-V1.0.safetensors']
QWEN_INPAINT_FILES = ['Qwen-Image-InstantX-ControlNet-Inpainting.safetensors', 'qwen_image_fp8_e4m3fn.safetensors']
FLUX_DEV_FILES = ['flux2_dev_fp8mixed.safetensors', 'Flux_2-Turbo-LoRA_comfyui.safetensors', 'mistral_3_small_flux2_bf16.safetensors']
FLUX_KLEIN_FILES = ['flux-2-klein-9b-kv-fp8.safetensors', 'flux-2-klein-9b-fp8.safetensors', 'flux-2-klein-base-9b-fp8.safetensors', 'qwen_3_8b_fp8mixed.safetensors']

def cleanup_other_workflows(keep_files):
    all_files = QWEN_EDIT_FILES + QWEN_CN_FILES + QWEN_INPAINT_FILES + FLUX_DEV_FILES + FLUX_KLEIN_FILES
    for f in all_files:
        if f not in keep_files:
            for folder in ['checkpoints', 'controlnet', 'loras', 'diffusion_models']:
                remove_if_exists(f'{MODELS_DIR}/{folder}/{f}')

if WORKFLOW == 'qwen_edit_2511':
    print('\U0001f3a8 Workflow: Qwen Image Edit 2511')
    cleanup_other_workflows(QWEN_EDIT_FILES)
    hf_download('Comfy-Org/Qwen-Image-Edit_ComfyUI', 'split_files/diffusion_models/qwen_image_edit_2511_bf16.safetensors', f'{MODELS_DIR}/diffusion_models')
    hf_download('lightx2v/Qwen-Image-Edit-2511-Lightning', 'Qwen-Image-Edit-2511-Lightning-4steps-V1.0-bf16.safetensors', f'{MODELS_DIR}/loras')
    hf_download('Comfy-Org/HunyuanVideo_1.5_repackaged', 'split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors', f'{MODELS_DIR}/text_encoders')
    hf_download('Comfy-Org/Qwen-Image_ComfyUI', 'split_files/vae/qwen_image_vae.safetensors', f'{MODELS_DIR}/vae')

elif WORKFLOW == 'qwen_controlnet':
    print('\U0001f3a8 Workflow: Qwen 2512 ControlNet Union')
    cleanup_other_workflows(QWEN_CN_FILES)
    hf_download('alibaba-pai/Qwen-Image-2512-Fun-Controlnet-Union', 'Qwen-Image-2512-Fun-Controlnet-Union-2602.safetensors', f'{MODELS_DIR}/controlnet')
    hf_download('Comfy-Org/Qwen-Image_ComfyUI', 'split_files/diffusion_models/qwen_image_2512_fp8_e4m3fn.safetensors', f'{MODELS_DIR}/diffusion_models')
    hf_download('lightx2v/Qwen-Image-Lightning', 'Qwen-Image-Lightning-4steps-V1.0.safetensors', f'{MODELS_DIR}/loras')
    hf_download('Comfy-Org/HunyuanVideo_1.5_repackaged', 'split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors', f'{MODELS_DIR}/text_encoders')
    hf_download('Comfy-Org/Qwen-Image_ComfyUI', 'split_files/vae/qwen_image_vae.safetensors', f'{MODELS_DIR}/vae')

elif WORKFLOW == 'qwen_inpainting':
    print('\U0001f3a8 Workflow: Qwen InstantX Inpainting')
    cleanup_other_workflows(QWEN_INPAINT_FILES)
    hf_download('Comfy-Org/Qwen-Image-InstantX-ControlNets', 'split_files/controlnet/Qwen-Image-InstantX-ControlNet-Inpainting.safetensors', f'{MODELS_DIR}/controlnet')
    hf_download('Comfy-Org/Qwen-Image_ComfyUI', 'split_files/diffusion_models/qwen_image_fp8_e4m3fn.safetensors', f'{MODELS_DIR}/diffusion_models')
    hf_download('Comfy-Org/Qwen-Image_ComfyUI', 'split_files/text_encoders/qwen_2.5_vl_7b_fp8_scaled.safetensors', f'{MODELS_DIR}/text_encoders')
    hf_download('Comfy-Org/Qwen-Image_ComfyUI', 'split_files/vae/qwen_image_vae.safetensors', f'{MODELS_DIR}/vae')

elif WORKFLOW == 'flux2_dev':
    print('\U0001f3a8 Workflow: FLUX.2-dev')
    cleanup_other_workflows(FLUX_DEV_FILES)
    hf_download('Comfy-Org/flux2-dev', 'split_files/diffusion_models/flux2_dev_fp8mixed.safetensors', f'{MODELS_DIR}/diffusion_models')
    hf_download('ByteZSzn/Flux.2-Turbo-ComfyUI', 'Flux_2-Turbo-LoRA_comfyui.safetensors', f'{MODELS_DIR}/loras')
    hf_download('Comfy-Org/flux2-dev', 'split_files/text_encoders/mistral_3_small_flux2_bf16.safetensors', f'{MODELS_DIR}/text_encoders')
    hf_download('black-forest-labs/FLUX.2-small-decoder', 'full_encoder_small_decoder.safetensors', f'{MODELS_DIR}/vae')

elif WORKFLOW == 'flux2_klein_kv':
    print('\U0001f3a8 Workflow: FLUX.2-klein 9B KV')
    cleanup_other_workflows(FLUX_KLEIN_FILES)
    hf_download('black-forest-labs/FLUX.2-klein-9b-kv-fp8', 'flux-2-klein-9b-kv-fp8.safetensors', f'{MODELS_DIR}/diffusion_models')
    hf_download('black-forest-labs/FLUX.2-klein-9b-fp8', 'flux-2-klein-9b-fp8.safetensors', f'{MODELS_DIR}/diffusion_models')
    hf_download('black-forest-labs/FLUX.2-klein-base-9b-fp8', 'flux-2-klein-base-9b-fp8.safetensors', f'{MODELS_DIR}/diffusion_models')
    hf_download('Comfy-Org/flux2-klein-9B', 'split_files/text_encoders/qwen_3_8b_fp8mixed.safetensors', f'{MODELS_DIR}/text_encoders')
    hf_download('Comfy-Org/flux2-dev', 'split_files/vae/flux2-vae.safetensors', f'{MODELS_DIR}/vae')
    hf_download('black-forest-labs/FLUX.2-small-decoder', 'full_encoder_small_decoder.safetensors', f'{MODELS_DIR}/vae')

elif WORKFLOW == 'flux2_klein_edit':
    print('\U0001f3a8 Workflow: FLUX.2-klein 9B Image Edit')
    cleanup_other_workflows(['flux-2-klein-base-9b-fp8.safetensors', 'qwen_3_8b_fp8mixed.safetensors'])
    hf_download('black-forest-labs/FLUX.2-klein-base-9b-fp8', 'flux-2-klein-base-9b-fp8.safetensors', f'{MODELS_DIR}/diffusion_models')
    hf_download('Comfy-Org/flux2-klein-9B', 'split_files/text_encoders/qwen_3_8b_fp8mixed.safetensors', f'{MODELS_DIR}/text_encoders')
    hf_download('black-forest-labs/FLUX.2-small-decoder', 'full_encoder_small_decoder.safetensors', f'{MODELS_DIR}/vae')

print('\n\u2705 Model indirme tamam')

---
# C) ComfyUI Başlat

In [ ]:
import subprocess
import time

import requests
from google.colab import userdata, output

USE_CLOUDFLARE = False

PORT = 8188

subprocess.run(['pkill', '-f', 'main.py'], capture_output=True)
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
time.sleep(2)

log_file = open('/content/comfyui.log', 'w')
comfy_proc = subprocess.Popen(
    ['python', 'main.py', '--listen', '0.0.0.0', '--port', str(PORT), '--gpu-only', '--enable-cors-header', '*'],
    cwd=COMFY_DIR,
    stdout=log_file,
    stderr=subprocess.STDOUT,
    stdin=subprocess.DEVNULL,
)
print(f'\U0001f680 ComfyUI başlatıldı (PID: {comfy_proc.pid})')

t0 = time.time()
ready = False
while time.time() - t0 < 120:
    try:
        if requests.get(f'http://localhost:{PORT}/system_stats', timeout=3).status_code == 200:
            ready = True
            break
    except requests.RequestException:
        pass
    if comfy_proc.poll() is not None:
        print('\u274c ComfyUI çöktü!')
        log_file.close()
        with open('/content/comfyui.log') as f:
            print(f.read()[-500:])
        break
    time.sleep(3)

if ready:
    print(f'\u2705 ComfyUI hazır ({int(time.time()-t0)}s)')
    if USE_CLOUDFLARE:
        token = userdata.get('CF_TUNNEL_TOKEN')
        cf_log = open('/content/cloudflared.log', 'w')
        cf_proc = subprocess.Popen(
            ['cloudflared', 'tunnel', '--no-autoupdate', 'run', '--token', token],
            stdout=cf_log, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
        )
        time.sleep(5)
        print(f'\U0001f310 Cloudflare: https://comfyui.ersamely.com')
    else:
        print(f'\U0001f310 Colab Proxy:')
        output.serve_kernel_port_as_window(PORT, path='/')
else:
    print('\u274c Timeout!')

In [ ]:
import time
from datetime import datetime, timezone

import requests

print('ComfyUI canlı tutma. Durdurmak için interrupt et.\n')
while True:
    try:
        local_ok = requests.get(f'http://localhost:{PORT}/system_stats', timeout=5).status_code == 200
    except requests.RequestException:
        local_ok = False
    comfy_alive = comfy_proc.poll() is None
    now = datetime.now(timezone.utc).strftime('%H:%M:%S UTC')
    c = '\u2705' if (local_ok and comfy_alive) else '\u274c'
    print(f'{now} | ComfyUI: {c}')
    time.sleep(30)